In [ ]:
import regex as re

special_tokens = ['<|endoftest|>']
special_pattern = "|".join(re.escape(t) for t in special_tokens)
chunk = 'aaa<|endoftest|>bbb'
chunks = re.split(special_pattern, chunk)
print(chunks)

In [1]:
import torch
import torch.nn as nn
from einops import rearrange, einsum


In [2]:
b = torch.ones((1, 5, 5))
a = torch.ones((5,5))
a


tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])

In [3]:
b

tensor([[[1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.]]])

In [4]:
a = torch.randn(3,2,5)
b = torch.randn(3,5,3)
print(a)
print(b)
torch.einsum('ijk,ikl->ijl', [a, b])

tensor([[[ 0.1289, -0.2533,  0.5849,  0.1867,  0.0032],
         [-0.3871,  0.8997, -1.0797, -0.7973, -0.0104]],

        [[-1.4799,  0.0726,  1.5355,  0.5673,  0.6732],
         [ 1.1563,  1.0249,  1.3684, -2.3241,  0.5438]],

        [[ 0.9264,  0.2767,  1.3078, -0.3672, -0.5333],
         [ 1.1950, -0.1672, -0.2352, -0.4829, -2.3205]]])
tensor([[[ 0.1514,  0.4684, -0.0897],
         [-0.1039, -0.6436, -0.0711],
         [ 0.1675,  0.7382, -0.8730],
         [ 0.4901, -1.5257, -2.0656],
         [-0.0076,  0.2798,  0.1748]],

        [[ 0.5677, -0.0313,  0.7685],
         [-1.2117, -0.0237, -0.8620],
         [-0.1408,  0.4297, -1.2719],
         [-0.8577,  0.5786, -0.6884],
         [ 1.3405, -0.0152, -0.0556]],

        [[-0.2017, -1.3797, -1.1980],
         [-1.1953,  1.8789, -2.7791],
         [ 0.7417, -1.1763,  0.9515],
         [ 0.7500, -0.2669,  0.0989],
         [-0.5633,  0.5439, -1.6596]]])


tensor([[[ 0.2353,  0.3713, -0.8892],
         [-0.7238, -0.3439,  2.5585]],

        [[-0.7285,  1.0224, -3.5808],
         [ 1.9443, -0.8254, -0.1656]],

        [[ 0.4774, -2.4885,  0.2143],
         [ 0.7295, -2.8194,  2.6126]]])

In [5]:
vocab_size = 50257
context_length = 1024
num_layers = 48
d_model = 1600
27
num_heads = 25
d_ff = 6400

emb = vocab_size * d_model
norm = d_model
att = d_model * d_model * 4
ff = d_model * d_ff * 3
block = norm * 2 + att + ff
block = block * num_layers
rope = 0
linear = d_model * vocab_size
ans = emb + block + rope + norm + linear
print(ans)
ans * 4 / 1000 / 1000/ 1000

2127057600


8.5082304

In [6]:
vocab_size = 50257
context_length = 1024
num_layers = 48
d_model = 1600
num_heads = 25
d_ff = 6400

token_emb = 0 # only table look ups
norm = 6 * context_length * d_model # pow, mean, +, rsqrt, *, *
att = 3 * (2 * d_model * d_model * context_length) # proj of qkv
att += d_model * context_length * context_length * 2 * 2 # qkv
att += d_model * context_length * d_model * 2 # proj of o
ff = 2 * context_length * d_model * d_ff * 3 # w1, w2, w3
block = norm * 2 + att + ff
block = block * num_layers
linear = 2 * context_length * d_model * vocab_size
ans = emb + block + rope + norm + linear
ans

print(att*num_layers/ans)
print(ff*num_layers/ans)
print(linear/ans)

0.2943390471991507
0.6689523799980698
0.03647953533155707


In [7]:
# flops of forward pass
vocab_size = 50257
context_length = 1024
num_layers = 48
d_model = 1600
num_heads = 25
d_ff = 6400

# num_layer = 12
# d_model = 768
# num_heads = 12

# num_layer = 24
# d_model = 1024
# num_heads = 16

# num_layer = 36
# d_model = 1280
# num_heads = 20

# context_length = 16384

token_emb = 0 # only table look ups
norm = 6 * context_length * d_model # pow, mean, +, rsqrt, *, *
att = 3 * (2 * d_model * d_model * context_length) # proj of qkv
att += d_model * context_length * context_length * 2 * 2 # qkv
att += d_model * context_length * d_model * 2 # proj of o
ff = 2 * context_length * d_model * d_ff * 3 # w1, w2, w3
block = norm * 2 + att + ff
block = block * num_layers
linear = 2 * context_length * d_model * vocab_size
ans = emb + block + rope + norm + linear
ans

print(f'total_flops={ans:,d}')
print(f'att_flops={att*num_layers:,d} flops_ratio={att*num_layers/ans:.2%}')
print(f'ff_flops={ff*num_layers:,d} flops_ratio={ff*num_layers/ans:.2%}')
print(f'linear_flops={linear:,d} flops_ratio={linear/ans:.2%}')

batch_size = 1024
steps = 400000
forward = batch_size * steps * ans
print(f'forward={forward:,d}')


total_flops=4,514,370,484,800
att_flops=1,328,755,507,200 flops_ratio=29.43%
ff_flops=3,019,898,880,000 flops_ratio=66.90%
linear_flops=164,682,137,600 flops_ratio=3.65%
forward=1,849,086,150,574,080,000,000


In [8]:
# peak memory of traning
vocab_size = 50257
context_length = 1024
num_layers = 48
d_model = 1600
num_heads = 25
d_ff = 6400
batch_size = 5

# parameters
emb = vocab_size * d_model
norm = d_model
att = d_model * d_model * 4
ff = d_model * d_ff * 2
block = norm * 2 + att + ff
block = block * num_layers
rope = 0
final_norm = d_model
linear = d_model * vocab_size
params = emb + block + rope + linear + final_norm

# activations
x = batch_size*context_length*d_model
norm = x
q = k = v = s = x
qk = batch_size*context_length*context_length*num_heads
att = x + q + k + v + qk + s
w1 = silu = batch_size*context_length*d_ff
ff = x + w1 + silu
block = att + ff + 2*norm
block = block * num_layers
final_norm = norm
loss_inputs = loss_log_probs = batch_size*context_length*vocab_size
loss = loss_inputs + loss_log_probs
activations = final_norm + block + loss

# gradients
gradients = params

# optimizer
m = v = params
optimizer = m + v

total = (params + activations + gradients + optimizer) * 4

print(f'total={total:,d}')

total=78,591,544,320


In [9]:
# flops of backward pass - optimizer

m = params * 3 # *, +, *
v = params * 4 # *, +, *, **2
pdata1 = params * 5 # *, ., **0.5, +, -
pdata2 = 2 # *, -
total = m + v + pdata1 + pdata2

optimizer = total * steps
print(f'optimizer={optimizer:,d}')

optimizer=7,850,580,480,800,000


In [10]:
# flops of backward pass - gradients (approximate)

backgrad = forward * 2
print(f'backgrad={backgrad:,d}')



backgrad=3,698,172,301,148,160,000,000


In [11]:
a100 = 19.5 * 1e12 * 0.5

total = forward + backgrad + optimizer

total/a100/60/60/24


6585.073958099146